In [ ]:
!pip install requests pandas

## Konsolidierte Importe erstellen

In [ ]:
import requests # Used for making HTTP requests to the Trello API and downloading images.
import pandas as pd # Used for data manipulation and creating DataFrames to store Trello card data.
import base64 # Used for encoding downloaded image content into base64 format for HTML embedding.
import mimetypes # Used to infer the MIME type of images when downloading, for proper base64 data URI formatting.
from IPython.display import display, HTML # Used to display the generated HTML content directly within the Jupyter Notebook.

## Globale Konfiguration definieren

In [ ]:
import os
import getpass
import requests
import pandas as pd

# --- Time range ----
YEAR = "2025" # either a year, or empty string = all years

# --- Trello API Credentials ---
# Do NOT hardcode secrets in this notebook. Provide them as environment
# variables (e.g. via a .env file, shell export, or Codespaces/Colab Secrets),
# or you will be prompted for them below.
# Create your own API Key/Token at: https://trello.com/power-ups/admin
API_KEY = os.environ.get("API_KEY") or input("Trello API Key: ")
TOKEN = os.environ.get("TOKEN") or getpass.getpass("Trello Token: ")
BOARD_ID = os.environ.get("BOARD_ID") or input("Trello Board ID: ")

# --- Custom Field IDs ---
# These IDs correspond to specific custom fields on your Trello board.
# They are used to identify and retrieve the values for 'Date' and 'Stars' from Trello cards.
# Adjust these to match the custom fields on your own board.
DATE_FIELD_ID = "650eab393bb23cef7cc22a01"
STARS_FIELD_ID = "650dd59d5973d39ae593be5b"

# --- Stars Options Mapping ---
# This dictionary maps the internal dropdown option IDs from the 'Stars' custom field in Trello
# to their human-readable and emoji representations. This ensures star ratings are displayed
# correctly and intuitively in the output.
stars_options = {
    "650e74f3185e03e3bdc91ad4": "🕳️", # Represents 'No Stars' or an invalid rating
    "650dd79347a7686c998bb170": "⭐",
    "650dd797372df4875df6ef91": "⭐⭐",
    "650dd79d3ab52f64293fb9fb": "⭐⭐⭐",
    "650dd7a1b752d20d0e181561": "⭐⭐⭐⭐",
    "650dd59d5973d39ae593be5c": "⭐⭐⭐⭐⭐"
}

print("Trello API credentials, custom field IDs, and stars_options mapping defined.")

## Funktion `fetch_list_names`


In [ ]:
def fetch_list_names(api_key, token, board_id):
    """
    Fetches all lists from a specified Trello board and creates a mapping of their IDs to names.

    Parameters:
    - api_key (str): Your Trello API key.
    - token (str): Your Trello API token.
    - board_id (str): The ID of the Trello board from which to fetch lists.

    Returns:
    - dict: A dictionary where keys are Trello list IDs (str) and values are list names (str).
            Returns an empty dictionary if an error occurs during the API request.
    """
    url = f"https://api.trello.com/1/boards/{board_id}/lists"
    params = {
        'key': api_key,
        'token': token
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        lists = response.json()
        return {list_['id']: list_['name'] for list_ in lists}
    else:
        print(f"Error fetching lists: {response.status_code}")
        return {}

print("Function 'fetch_list_names' defined.")

## Funktion `fetch_cards_with_covers_and_custom_fields`

In [ ]:
def fetch_cards_with_covers_and_custom_fields(api_key, token, board_id):
    """
    Fetches all cards from a specified Trello board, including their names, descriptions,
    cover images, and custom field values (Date and Stars). Cards are filtered to include
    only those with a 'Date' or a 'Stars'.

    The function first retrieves all lists on the board to map list IDs to names.
    For each card, it attempts to find a cover image, prioritizing attachments explicitly
    marked as a cover, then falling back to the largest preview of the first attachment.
    It extracts custom field values for 'Date' and 'Stars' using predefined field IDs and mappings.

    Parameters:
    - api_key (str): Your Trello API key.
    - token (str): Your Trello API token.
    - board_id (str): The ID of the Trello board from which to fetch cards.

    Returns:
    - list: A list of dictionaries, where each dictionary represents a filtered card.
            Each card dictionary contains:
            - 'List' (str): The name of the list the card belongs to.
            - 'Title' (str): The name of the card.
            - 'Date' (str, optional): The value of the 'Date' custom field.
            - 'Stars' (str, optional): The mapped value of the 'Stars' custom field (e.g., '⭐⭐⭐').
            - 'Description' (str): The description of the card.
            - 'Image' (str, optional): The URL of the card's cover image.
            - 'Year' (str, optional): The extracted year from the 'Date' field.
            Returns an empty list if an error occurs during the API request or no cards meet the criteria.
    """
    # Fetch list names to map card's list_id to human-readable list names
    list_names = fetch_list_names(api_key, token, board_id)

    # Construct the API URL for fetching cards from the specified board.
    # 'attachments' and 'customFieldItems' are set to 'true' to include cover images
    # and custom field data in the response.
    url = f"https://api.trello.com/1/boards/{board_id}/cards"
    params = {
        'key': api_key,
        'token': token,
        'attachments': 'true',
        'fields': 'name,desc,idList,customFieldItems',
        'customFieldItems': 'true'
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        cards = response.json()
        data = [] # Initialize a list to store processed card data
        for card in cards:
            cover = None
            custom_fields = {} # Dictionary to hold extracted custom field values

            # Process attachments to find the cover image.
            # Prioritize attachments explicitly marked as cover, otherwise use largest preview.
            if 'attachments' in card:
                for attachment in card['attachments']:
                    if attachment.get('isCover'):
                        cover = attachment['url']
                        break
                    elif 'previews' in attachment and attachment['previews']:
                        cover = attachment['previews'][-1]['url']
                        break

            # Extract custom field values for 'Date' and 'Stars' using their predefined IDs.
            if 'customFieldItems' in card:
                for custom_field in card['customFieldItems']:
                    # Check for the Date custom field and extract its text value
                    if custom_field['idCustomField'] == DATE_FIELD_ID and custom_field.get('value'):
                        custom_fields['Date'] = custom_field['value'].get('text')
                    # Check for the Stars custom field and map its ID value to the defined star options
                    elif custom_field['idCustomField'] == STARS_FIELD_ID and custom_field.get('idValue'):
                        custom_fields['Stars'] = stars_options.get(custom_field['idValue'], None)

            # Extract year from 'Date' if available
            card_year = None
            if custom_fields.get('Date') and len(custom_fields['Date']) >= 4:
                try:
                    card_year = custom_fields['Date'][:4]
                except:
                    card_year = None # In case date format is unexpected

            # check time-range
            if YEAR: # eg 2025
                if card_year != YEAR:
                    continue

            # If neither 'Date' nor 'Stars' custom fields have values, skip this card
            if not (custom_fields.get('Date') or custom_fields.get('Stars')):
                continue

            # Retrieve the list name using the mapped list_names dictionary.
            list_name = list_names.get(card.get('idList', ''), "Unknown List")

            # If Stars is empty
            if not custom_fields.get('Stars'):
                custom_fields['Stars'] = ""

            # if Description is empty
            if not card['desc']:
                card['desc'] = "-"

            # Append the extracted and processed card data to our list.
            data.append({
                'List': list_name,
                'Title': card['name'],
                'Date': custom_fields.get('Date'),
                'Stars': custom_fields.get('Stars'),
                'Description': card['desc'],
                'Image': cover,
                'Year': card_year # Add the extracted year
            })
        return data
    else:
        print(f"Error: {response.status_code}")
        return []

print("Function 'fetch_cards_with_covers_and_custom_fields' defined.")

## Funktion `get_base64_image`

In [ ]:
!pip install Pillow

In [ ]:
import io
from PIL import Image # Import the Pillow library for image processing

def get_base64_image(image_url, max_width=600, quality=100):
    """
    Downloads an image from a given URL, resizes it, and converts it into a base64 encoded string.
    This format is suitable for embedding directly into HTML documents using data URIs.

    Parameters:
    - image_url (str): The URL of the image to be processed.
    - max_width (int): The maximum width for the image. The height will be adjusted proportionally.
    - quality (int): The compression quality for JPEG images (0-100). Lower values mean smaller size.

    Returns:
    - str: A base64 encoded string prefixed with a data URI (e.g., 'data:image/jpeg;base64,...')
           if the image is successfully downloaded, resized, and encoded.
    - None: If the image_url is invalid, empty, or if an error occurs during download, resizing or encoding.
    """
    if pd.isna(image_url) or not image_url.strip():
        return None
    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()

        if not response.content:
            print(f"Warning: Downloaded content for {image_url} is empty. Cannot embed image.")
            return None

        # Use Pillow to open and resize the image
        image_data = io.BytesIO(response.content)
        img = Image.open(image_data)

        # Resize image while maintaining aspect ratio
        width, height = img.size
        if width > max_width:
            new_height = int(max_width * height / width)
            img = img.resize((max_width, new_height), Image.Resampling.LANCZOS)

        # Save the resized image to a bytes buffer with compression
        buffered = io.BytesIO()
        # Try to save as original format, fall back to JPEG if not supported or no specific format
        image_format = img.format if img.format and img.format in ['JPEG', 'PNG', 'GIF', 'WEBP'] else 'JPEG'

        # Convert RGBA to RGB if saving as JPEG, as JPEG does not support transparency
        if img.mode == 'RGBA' and image_format == 'JPEG':
            img = img.convert('RGB')

        img.save(buffered, format=image_format, quality=quality) # Apply compression quality
        encoded_image = base64.b64encode(buffered.getvalue()).decode('utf-8')

        # Infer content type from the final image format
        content_type = f"image/{image_format.lower()}"

        return f"data:{content_type};base64,{encoded_image}"
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image {image_url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred for {image_url}: {e}")
        return None

print("Function 'get_base64_image' updated with image resizing and compression.")

## Hauptdatenverarbeitungslogik und CSV speichern

In [ ]:
import pandas as pd # Ensure pandas is imported if not already in this cell's context.

# Fetch data from Trello using the defined function.
# This call retrieves card information, including custom fields and cover images.
cards_data = fetch_cards_with_covers_and_custom_fields(API_KEY, TOKEN, BOARD_ID)

# Convert the fetched list of dictionaries into a Pandas DataFrame for easier manipulation.
df = pd.DataFrame(cards_data)
print("Trello cards loaded.")

# Apply the function to the 'Image' column to create 'EmbeddedImage'.
# This step converts image URLs to base64 encoded strings for direct embedding in HTML.
df['EmbeddedImage'] = df['Image'].apply(get_base64_image)
print("Embedded images loaded.")

# Sort the DataFrame by the 'Date' column in descending order, if the column exists.
# This ensures that events are listed from newest to oldest in the output.
if "Date" in df.columns:
    df = df.sort_values(by="Date", ascending=False)

if YEAR == "": # only if time-range not restricted
    # Save the processed DataFrame to a CSV file named "Theater & Kultur Reviews.csv".
    # 'index=False' prevents Pandas from writing the DataFrame index as a column in the CSV.
    # Exclude the 'EmbeddedImage' column from the CSV export.
    df.drop(columns=['EmbeddedImage']).to_csv("Theater & Kultur Reviews.csv", index=False)
    print("CSV exported, needs to be imported into Excel.")

## HTML-Generierung und Anzeige

In [ ]:
from IPython.display import display, HTML

# Initialize an empty string variable to store the generated HTML content.
html_output_embedded = ""

# Apply the image base64 encoding function to the 'Image' column to create 'EmbeddedImage'.
# This step ensures that image data is ready for embedding and addresses potential
# 'KeyError' if the DataFrame was reloaded without this column being generated.
df['EmbeddedImage'] = df['Image'].apply(get_base64_image)

# Embed Google Fonts for a Garamond-like font ('EB Garamond') and define CSS styling.
# This style block ensures consistent font usage and robustly removes borders from tables and cells.
html_output_embedded += """
<style>
@import url('https://fonts.googleapis.com/css2?family=EB+Garamond:wght@400;700&display=swap');
body { font-family: 'EB Garamond', serif; }
table, th, td {
    border: none !important; /* Force no borders for all table elements */
}
table {
    border-collapse: collapse; /* Collapse table borders to prevent double borders */
    width: 100%; /* Make the table full width */
    margin-bottom: 20px; /* Add some space below each table */
}
</style>
"""

# Define the custom order for lists. This order dictates how headings and cards
# will appear in the final HTML output.
list_order = ["Theater", "Tanz, Musical, Konzert", "Kabarett", "Ausstellung"]

# Iterate through each list name in the predefined custom order.
# This loop processes cards list by list, ensuring proper sectioning in the HTML.
for list_name in list_order:
    # Filter the DataFrame to get only the rows (cards) belonging to the current list_name.
    list_df = df[df['List'] == list_name]

    # If no cards are found for the current list, skip to the next list to avoid empty sections.
    if list_df.empty:
        continue

    # Get the total count of cards in the current main section.
    card_count = len(list_df)

    # Append an <h3> heading for the current list, including the card count.
    # Custom styling is applied for font family and size to match the overall design.
    html_output_embedded += f"<h3 style=\"font-family: 'EB Garamond', serif; font-size: 1.8em;\">{list_name} ({card_count})</h3>"

    # Conditionally add year subsections if no specific year is selected (YEAR == "")
    if YEAR == "":
        # Get unique sorted years for this list, ensuring no NaNs and sorting in descending order.
        unique_years = sorted(list_df['Year'].dropna().unique(), reverse=True)

        # Iterate through each year to create sub-sections.
        for year in unique_years:
            # Filter the list_df further for the current year.
            year_df = list_df[list_df['Year'] == year]

            # If no cards are found for the current year, skip to the next year.
            if year_df.empty:
                continue

            # Get the count of cards in the current year sub-section.
            year_card_count = len(year_df)

            # Append an <h4> heading for the current year sub-section.
            html_output_embedded += f"<h4 style=\"font-family: 'EB Garamond', serif; font-size: 1.4em; margin-top: 15px;\">{year} ({year_card_count})</h4>"

            # Append the opening <table> and <tbody> tags for the year's cards.
            html_output_embedded += """
            <table style="font-family: 'EB Garamond', serif;">
            <tbody>
            """

            # Iterate through each row (card) in the filtered DataFrame for the current year.
            # This loop generates the HTML for each individual card's details.
            for _, row in year_df.iterrows():
                # For each card, append a <tr> (table row) with three <td> (table data) cells.
                # Each <td> contains specific card information (Title, Date, Stars, Image, Description).
                # Inline styling 'border: none;' is added for maximum robustness in ensuring no borders,
                # and 'vertical-align: top;' aligns content to the top of the cell for better readability.
                html_output_embedded += f"""
                <tr>
                  <td style="width: 20%; vertical-align: top; padding: 8px; border: none;">
                    <strong>{row['Title']}</strong><br />
                    <span>{row['Date']}</span><br />
                    <span>{row['Stars']}</span>
                  </td>
                  <td style="width: 20%; vertical-align: top; text-align: center; padding: 8px; border: none;">
                    <img src="{row['EmbeddedImage']}" alt="No Image" style="width: 150px; height: auto;" />
                  </td>
                  <td style="width: 60%; vertical-align: top; padding: 8px; border: none;">{row['Description']}</td>
                </tr>
                """

            # After all cards for a year are processed, append the closing <tbody> and </table> tags.
            html_output_embedded += """
              </tbody>
            </table>
            """
    else: # If a specific year is selected (YEAR is not empty), display all cards for the list directly
        # Append the opening <table> and <tbody> tags for the list's cards.
        html_output_embedded += """
        <table style="font-family: 'EB Garamond', serif;">
        <tbody>
        """
        # Iterate through each row (card) in the filtered DataFrame for the current list.
        for _, row in list_df.iterrows():
            html_output_embedded += f"""
            <tr>
              <td style="width: 20%; vertical-align: top; padding: 8px; border: none;">
                <strong>{row['Title']}</strong><br />
                <span>{row['Date']}</span><br />
                <span>{row['Stars']}</span>
              </td>
              <td style="width: 20%; vertical-align: top; text-align: center; padding: 8px; border: none;">
                <img src="{row['EmbeddedImage']}" alt="No Image" style="width: 150px; height: auto;" />
              </td>
              <td style="width: 60%; vertical-align: top; padding: 8px; border: none;">{row['Description']}</td>
            </tr>
            """
        # After all cards for a list are processed, append the closing <tbody> and </table> tags.
        html_output_embedded += """
          </tbody>
        </table>
        """

# Display the complete generated HTML content in the notebook output.
# This renders the structured Trello card data directly within the Colab environment.
display(HTML(html_output_embedded))

print("HTML table with embedded images generated and displayed successfully.")

## HTML-Datei speichern

In [ ]:
import os # Ensure os module is imported for file operations, if not already. This is generally good practice.

# Save the generated HTML content to an HTML file.
# This allows the Trello board review to be viewed as a standalone webpage.
if YEAR == "": filename = "Theater & Kultur Reviews.html"
else: filename = f"{YEAR} Theater & Kultur.html"

with open(filename, "w", encoding="utf-8") as f:
    f.write(html_output_embedded)

print("Table with embedded images saved as HTML, need to convert to PDF.")